In [ ]:
# base setup to configure Model and agents
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from watchfiles import awatch

model = OpenAIChatModel(
    model_name="llama3:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(model=model, system_prompt="You are a helpful assistant")

result = await agent.run("What is th capital of Marche region in italy?")
print(result.output)

In [ ]:
# using Pydantic Ai UI to interact with the model
import uvicorn
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

model = OpenAIChatModel(
    model_name="llama3:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(model=model, system_prompt="You are a helpful assistant")
app = agent.to_web()

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

In [ ]:
# structured output using mistral

from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

class CityModel(BaseModel):
    city: str
    country: str
    population: int
    fun_fact: str


model = OpenAIChatModel(
    model_name="mistral:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(
    model=model,
    output_type=CityModel,
    system_prompt="You are a geography expert."
)

result = await agent.run("Tell me about rome in italy")
print(result.output.city)
print(result.output.population)


In [ ]:
# tools
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai import Agent
from datetime import datetime


model = OpenAIChatModel(
    model_name="qwen2.5:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(
    model=model,
    system_prompt="You are a geography expert."
)

@agent.tool_plain
def get_current_time() -> str:
    """Returns the current time."""
    return datetime.now().strftime("%H:%M:%S")


@agent.tool_plain
def get_weather(city: str) -> str:
    """Gets the weather for a city."""
    # mock - replace with real API
    return f"It's sunny and 22°C in {city}"

result = await agent.run("What time is it and what's the weather in Rome?")
print(result.output)

In [ ]:
# message history
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai import Agent

model = OpenAIChatModel(
    model_name="qwen2.5:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(model=model)

# First turn
result1 = await agent.run("My name is Antonio")
print(f"Turn 1: {result1.output}")

history = result1.all_messages()  # 👈 was result, should be result1

# Second turn
result2 = await agent.run("What is my name?", message_history=history)
print(f"Turn 2: {result2.output}")  # 👈 was result, should be result2

In [ ]:
# dependency injection

from pydantic_ai import RunContext
from dataclasses import dataclass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

@dataclass
class UserContext:
    username: str
    is_premium: bool

model = OpenAIChatModel(
    model_name="qwen2.5:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(
    model=model,
    deps_type=UserContext,
    system_prompt="You are a helpful assistant."
)


@agent.system_prompt
def personalize(ctx: RunContext[UserContext]) -> str:
    tier = "premium" if ctx.deps.is_premium else "free"
    return f"The user is {ctx.deps.username} on the {tier} plan."

user = UserContext(username="Antonio", is_premium=True)
result = await agent.run("What can i do?", deps=user)
print(result.output)


In [ ]:
# streaming response
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai import Agent

model = OpenAIChatModel(
    model_name="qwen2.5:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(model=model)

async with agent.run_stream("Write a short poem about Rome") as response:
    async for chunk in response.stream_text():
        print(chunk, end="", flush=True)
